In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile arc_gan_rl_solution.py
import os
import json
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import random
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold

from pathlib import Path
from tqdm import tqdm
import pickle # Import pickle for saving/loading processed data

# --- 1. CONFIGURATION ---
CONFIG = {
    # Data Paths
    # Manually verify and correct the path below based on your Google Drive structure.
    # This path is an example and likely needs adjustment.
    'base_data_path': '/content/drive/MyDrive/Google_AI_Studio/ARCAGI2025/data', # Corrected base path based on initial notebook mount
    'input_directory': 'GridTransitionDataset/training_transformed_unique_ids',
    'processed_data_file': 'processed_sequences.pkl', # Added processed data file name
    'output_directory': '/content/drive/MyDrive/Google_AI_Studio/ARCAGI2025/models',
    'rl_output_directory': '/content/drive/MyDrive/Google_AI_Studio/ARCAGI2025/rl_results', # Added RL output directory
    'submission_file': '/content/drive/MyDrive/Google_AI_Studio/ARCAGI2025/submission.json', # Added submission file path


    # Model Hyperparameters
    'vocab_size': 12,
    'max_seq_len': 1802, # Adjusted based on common ARC task grid sizes (max 30x30 input, 30x30 output, start/sep) -> (900 + 900 + 2) = 1802 max
    'd_model': 128,         # adjustable
    'nhead': 4,
    'num_layers': 4,        # adjustable
    'dim_feedforward': 512, # adjustable
    'dropout': 0.1,

    # GAN Training Parameters
    'gan_batch_size': 16, # adjustable
    'gan_learning_rate': 0.0002,
    'gan_num_epochs': 10,       # adjustable
    'n_splits': 5,
    'train_generator_every': 1,
    'label_smoothing': 0.1, # Added label smoothing parameter
    'gradient_clipping_value': 1.0, # Added gradient clipping parameter
    'gradient_accumulation_steps': 4, # Added gradient accumulation steps
    'checkpoint_interval': 1, # Save checkpoint every X epochs

    # RL Training Parameters
    'rl_learning_rate': 0.0001,
    'rl_num_episodes': 1000, # adjustable
    'gamma': 0.99, # Discount factor
    'epsilon_start': 1.0,
    'epsilon_end': 0.01,
    'epsilon_decay': 0.995,
    'rl_batch_size': 32, # Batch size for RL agent training
    'rl_checkpoint_interval': 100, # Save RL checkpoint every X episodes
    'rl_warmup_episodes': 100, # Episodes to collect experience before training


    # Environment Parameters (for RL)
    # These will depend on how you define your RL environment and reward function.
    # Example:
    # 'max_steps_per_episode': 100,
    # 'reward_success': 10,
    # 'reward_failure': -1,
    # 'reward_step': -0.01,
}

# --- 2. DATASET CLASS ---
class ARCDataset(Dataset):
    def __init__(self, sequences):
        """
        Args:
            sequences (list of torch.Tensor): A list of padded sequences.
        """
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]

# --- 3. MODEL ARCHITECTURE ---
class Generator(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, dropout):
        super().__init__()
        # Embedding layer to convert discrete tokens to vectors
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # Transformer Encoder to process the embedded sequence
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        # Output layer to convert back to token logits
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, src):
        # src is expected to be a LongTensor of token indices
        src = self.token_embedding(src)
        output = self.transformer_encoder(src)
        logits = self.output_layer(output)
        return logits

class Discriminator(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, dropout):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, src):
        src = self.token_embedding(src)
        output = self.transformer_encoder(src)
        # Use mean pooling across the sequence dimension
        output = torch.mean(output, dim=1)
        return self.output_layer(output)

# --- 4. WEIGHTS INITIALIZATION ---
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Embedding):
        nn.init.xavier_uniform_(m.weight)

# --- 5. DATA PREPROCESSING, CACHING, AND SAVING ---
def preprocess_and_cache_data(config):
    """
    Loads, preprocesses, and caches data from JSON files.
    Saves the processed data to a file.
    """
    processed_data_path = Path(config['base_data_path']) / config['processed_data_file']

    if processed_data_path.exists():
        print(f"Loading processed data from {processed_data_path}")
        try:
            with open(processed_data_path, 'rb') as f:
                all_sequences = pickle.load(f)
            print(f"Loaded {len(all_sequences)} sequences from cache.")
            return all_sequences
        except Exception as e:
            print(f"Error loading processed data from {processed_data_path}: {e}")
            print("Attempting to re-process data.")
            # Continue to reprocessing if loading fails


    print("--- Pre-processing and Caching All Data ---")

    data_dir = os.path.join(config['base_data_path'], config['input_directory'])
    print(f"Base data path: {config['base_data_path']}")
    print(f"Input directory: {config['input_directory']}")
    full_data_path = os.path.join(config['base_data_path'], config['input_directory'])
    print(f"Full data directory path: {full_data_path}")

    if not os.path.exists(full_data_path):
        print(f"Error: Data directory not found at {full_data_path}")
        return [] # Return empty list if directory doesn't exist

    # Verify if the data directory contains any files or subdirectories
    # Check if the path is a directory and contains anything
    if not os.path.isdir(full_data_path) or not any(os.scandir(full_data_path)):
         print(f"Warning: Data directory {full_data_path} is empty or not a valid directory.")
         return []


    glob_pattern = os.path.join(full_data_path, '**', '*.json')
    print(f"Using glob pattern: {glob_pattern}")

    all_file_paths = list(glob.iglob(glob_pattern, recursive=True))

    if not all_file_paths:
        print(f"Error: No JSON files found using pattern {glob_pattern}")
        return [] # Return empty list if no files are found


    all_sequences = []
    start_token, sep_token, pad_token = 10, 11, 0

    print(f"Found {len(all_file_paths)} files. Processing...")

    for file_path in tqdm(all_file_paths, desc="Loading and Caching Data"):
        # print(f"Processing file: {file_path}") # Uncomment for detailed file processing logs
        try:
            with open(file_path, 'r') as f:
                task = json.load(f)
                if 'train' in task and task['train']:
                    for example in task['train']:
                        if 'input' in example and example['input'] and 'output' in example and example['output']:
                            input_grid = np.array(example['input'])
                            output_grid = np.array(example['output'])

                            input_flat = input_grid.flatten()
                            output_flat = output_grid.flatten()

                            sequence = np.concatenate([
                                [start_token],
                                input_flat,
                                [sep_token],
                                output_flat
                            ])

                            # Pad or truncate sequence to max length
                            if len(sequence) > config['max_seq_len']:
                                sequence = sequence[:config['max_seq_len']]

                            padded_sequence = np.pad(sequence, (0, config['max_seq_len'] - len(sequence)), 'constant', constant_values=pad_token)
                            all_sequences.append(torch.tensor(padded_sequence, dtype=torch.long))
                        else:
                            pass # Skipping example due to missing or empty 'input' or 'output'.
                else:
                    pass # Skipping file due to missing or empty 'train' key.
        except json.JSONDecodeError:
            print(f"Error decoding JSON from file: {file_path}")
            continue
        except Exception as e:
            print(f"Error processing file {file_path}: {e}")
            continue

    print(f"Total sequences processed: {len(all_sequences)}")

    # Save processed data
    if all_sequences: # Only save if there are sequences to save
        processed_data_path.parent.mkdir(parents=True, exist_ok=True)
        with open(processed_data_path, 'wb') as f:
            pickle.dump(all_sequences, f)
        print(f"Processed data saved to {processed_data_path}")
    else:
        print("No sequences processed, skipping saving data.")


    return all_sequences


# --- 6. GAN TRAINER CLASS ---
class GANTrainer:
    def __init__(self, config, all_sequences):
        self.config = config
        self.all_sequences = all_sequences
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        os.makedirs(self.config['output_directory'], exist_ok=True)
        print(f"GAN models will be saved to: {self.config['output_directory']}")


        # Initialize final_generator to None
        self.final_generator = None


    def save_checkpoint(self, generator, discriminator, optimizer_G, optimizer_D, scheduler_G, scheduler_D, epoch, fold):
        """Saves a training checkpoint."""
        checkpoint_path = os.path.join(self.config['output_directory'], f'checkpoint_fold_{fold}_epoch_{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'scheduler_G_state_dict': scheduler_G.state_dict(),
            'scheduler_D_state_dict': scheduler_D.state_dict(),
        }, checkpoint_path)
        print(f"Checkpoint saved for Fold {fold}, Epoch {epoch} to {checkpoint_path}") # Keep print for verification

    def load_checkpoint(self, generator, discriminator, optimizer_G, optimizer_D, scheduler_G, scheduler_D, fold):
        """Loads the latest checkpoint for a given fold."""
        checkpoint_files = glob.glob(os.path.join(self.config['output_directory'], f'checkpoint_fold_{fold}_*.pth'))
        if not checkpoint_files:
            print(f"No checkpoint found for Fold {fold}. Starting from scratch.")
            return 0

        # Find the latest checkpoint based on epoch number in the filename
        latest_checkpoint = max(checkpoint_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
        print(f"Loading checkpoint: {latest_checkpoint}")

        try:
            checkpoint = torch.load(latest_checkpoint, map_location=self.device)
            generator.load_state_dict(checkpoint['generator_state_dict'])
            discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
            optimizer_G.load_state_dict(checkpoint['optimizer_G_state_dict'])
            optimizer_D.load_state_dict(checkpoint['optimizer_D_state_dict'])
            scheduler_G.load_state_dict(checkpoint['scheduler_G_state_dict'])
            scheduler_D.load_state_dict(checkpoint['scheduler_D_state_dict'])

            # Remove older checkpoints to save space
            for old_checkpoint in checkpoint_files:
                if old_checkpoint != latest_checkpoint:
                    try:
                        os.remove(old_checkpoint)
                        # print(f"Removed old checkpoint: {old_checkpoint}") # Commented out for cleaner output
                    except OSError as e:
                        print(f"Error removing old checkpoint {old_checkpoint}: {e}")

            return checkpoint['epoch']
        except Exception as e:
            print(f"Error loading checkpoint {latest_checkpoint}: {e}")
            print("Starting training from scratch.")
            return 0


    def train_one_fold(self, fold, train_loader):
        """Trains the GAN for one fold."""
        print(f"Initializing models for Fold {fold+1}...")
        generator = Generator(
            self.config['vocab_size'], self.config['d_model'], self.config['nhead'],
            self.config['num_layers'], self.config['dim_feedforward'], self.config['dropout']
        ).to(self.device)
        discriminator = Discriminator(
            self.config['vocab_size'], self.config['d_model'], self.config['nhead'],
            self.config['num_layers'], self.config['dim_feedforward'], self.config['dropout']
        ).to(self.device)

        optimizer_G = optim.AdamW(generator.parameters(), lr=self.config['gan_learning_rate'])
        optimizer_D = optim.AdamW(discriminator.parameters(), lr=self.config['gan_learning_rate'])
        criterion = nn.BCEWithLogitsLoss()

        scheduler_G = CosineAnnealingLR(optimizer_G, T_max=self.config['gan_num_epochs'])
        scheduler_D = CosineAnnealingLR(optimizer_D, T_max=self.config['gan_num_epochs'])

        scaler = GradScaler()

        start_epoch = self.load_checkpoint(generator, discriminator, optimizer_G, optimizer_D, scheduler_G, scheduler_D, fold + 1)

        if start_epoch > 0:
            print(f"Resuming training from epoch {start_epoch + 1}")
        else:
            generator.apply(weights_init)
            discriminator.apply(weights_init)
            print("Models initialized with weights_init.")

        fold_disc_losses_epoch = []
        fold_gen_losses_epoch = []

        print(f"Starting training for Fold {fold+1}...")
        for epoch in range(start_epoch, self.config['gan_num_epochs']):
            generator.train()
            discriminator.train()
            epoch_disc_losses = []
            epoch_gen_losses = []

            for i, real_sequences in enumerate(tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{self.config['gan_num_epochs']}")):
                real_sequences = real_sequences.to(self.device)
                batch_size = real_sequences.size(0)

                smooth_real_labels = torch.full((batch_size, 1), 1.0 - self.config['label_smoothing'], device=self.device)
                smooth_fake_labels = torch.full((batch_size, 1), self.config['label_smoothing'], device=self.device)

                # --- Train Discriminator ---
                optimizer_D.zero_grad()
                with autocast():
                    # Train with real data
                    d_output_real = discriminator(real_sequences)
                    d_loss_real = criterion(d_output_real, smooth_real_labels)

                    # Train with fake data
                    # Generate random input for the generator
                    fake_input_indices = torch.randint(0, self.config['vocab_size'], (batch_size, self.config['max_seq_len']), dtype=torch.long, device=self.device)
                    fake_sequences_logits = generator(fake_input_indices)
                    # Use Gumbel-Softmax to get differentiable discrete samples
                    fake_sequences = torch.argmax(F.gumbel_softmax(fake_sequences_logits, tau=0.5, hard=True, dim=-1), dim=-1)

                    d_output_fake = discriminator(fake_sequences.detach()) # Detach to not train generator
                    d_loss_fake = criterion(d_output_fake, smooth_fake_labels)

                    d_loss = d_loss_real + d_loss_fake

                scaler.scale(d_loss).backward()
                if (i + 1) % self.config['gradient_accumulation_steps'] == 0:
                    scaler.unscale_(optimizer_D)
                    torch.nn.utils.clip_grad_norm_(discriminator.parameters(), self.config['gradient_clipping_value'])
                    scaler.step(optimizer_D)
                    scaler.update()
                    optimizer_D.zero_grad()

                epoch_disc_losses.append(d_loss.item())

                # --- Train Generator ---
                if (i + 1) % self.config['train_generator_every'] == 0:
                    optimizer_G.zero_grad()
                    with autocast():
                        gen_labels = torch.ones(batch_size, 1, device=self.device)

                        fake_input_indices_gen = torch.randint(0, self.config['vocab_size'], (batch_size, self.config['max_seq_len']), dtype=torch.long, device=self.device)
                        fake_sequences_logits_gen = generator(fake_input_indices_gen)
                        fake_sequences_gen = torch.argmax(F.gumbel_softmax(fake_sequences_logits_gen, tau=0.5, hard=True, dim=-1), dim=-1)


                        g_output = discriminator(fake_sequences_gen)
                        g_loss = criterion(g_output, gen_labels)

                    scaler.scale(g_loss).backward()

                    if (i + 1) % self.config['gradient_accumulation_steps'] == 0:
                        scaler.unscale_(optimizer_G)
                        torch.nn.utils.clip_grad_norm_(generator.parameters(), self.config['gradient_clipping_value'])
                        scaler.step(optimizer_G)
                        scaler.update()
                        optimizer_G.zero_grad()

                    epoch_gen_losses.append(g_loss.item())


            scheduler_G.step()
            scheduler_D.step()

            avg_epoch_disc_loss = sum(epoch_disc_losses) / len(epoch_disc_losses) if epoch_disc_losses else 0
            avg_epoch_gen_loss = sum(epoch_gen_losses) / len(epoch_gen_losses) if epoch_gen_losses else 0
            fold_disc_losses_epoch.append(avg_epoch_disc_loss)
            fold_gen_losses_epoch.append(avg_epoch_gen_loss)

            print(f"Fold {fold+1}, Epoch {epoch+1}: Avg Discriminator Loss = {avg_epoch_disc_loss:.4f}, Avg Generator Loss = {avg_epoch_gen_loss:.4f}")

            # Save checkpoint at the end of each epoch
            if (epoch + 1) % self.config['checkpoint_interval'] == 0:
                self.save_checkpoint(generator, discriminator, optimizer_G, optimizer_D, scheduler_G, scheduler_D, epoch + 1, fold + 1)


        avg_fold_discriminator_loss = sum(fold_disc_losses_epoch) / len(fold_disc_losses_epoch) if fold_disc_losses_epoch else 0
        avg_fold_generator_loss = sum(fold_gen_losses_epoch) / len(fold_gen_losses_epoch) if fold_gen_losses_epoch else 0

        print(f"--- Finished Fold {fold+1} ---")
        print(f"Average Discriminator Loss: {avg_fold_discriminator_loss:.4f}")
        print(f"Average Generator Loss: {avg_fold_gen_loss:.4f}")

        # Save the final generator for this fold
        self.save_model(generator, f'generator_fold_{fold+1}.pth')


        return avg_fold_discriminator_loss, avg_fold_gen_loss

    def run_cross_validation(self):
        k_fold = KFold(n_splits=self.config['n_splits'], shuffle=True, random_state=42)

        fold_discriminator_losses = []
        fold_generator_losses = []

        if len(self.all_sequences) == 0:
            print("No sequences loaded. Cannot perform cross-validation.")
            return

        dataset = ARCDataset(self.all_sequences)

        print(f"--- Starting K-Fold Cross-Validation with {self.config['n_splits']} folds ---")

        # Get file paths for splitting (using index is safer with KFold)
        all_indices = list(range(len(self.all_sequences)))

        for fold, (train_ids, val_ids) in enumerate(k_fold.split(all_indices)):
            print(f"\n--- FOLD {fold+1}/{self.config['n_splits']} ---")

            train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
            # val_subsampler = torch.utils.data.SubsetRandomSampler(val_ids) # Validation not used in this GAN training loop

            train_loader = DataLoader(dataset, batch_size=self.config['gan_batch_size'], sampler=train_subsampler, num_workers=os.cpu_count() // 2 if os.cpu_count() else 0)

            avg_fold_disc_loss, avg_fold_gen_loss = self.train_one_fold(
                fold, train_loader
            )

            fold_discriminator_losses.append(avg_fold_disc_loss)
            fold_generator_losses.append(avg_fold_gen_loss)

        print("\n--- Cross-Validation Results ---")
        avg_disc_loss_cv = sum(fold_discriminator_losses) / self.config['n_splits'] if fold_discriminator_losses else 0
        avg_gen_loss_cv = sum(fold_generator_losses) / self.config['n_splits'] if fold_generator_losses else 0
        print(f"Average Discriminator Loss across {self.config['n_splits']} folds: {avg_disc_loss_cv:.4f}")
        print(f"Average Generator Loss across {self.config['n_splits']} folds: {avg_gen_loss_cv:.4f}")
        self.save_results(fold_discriminator_losses, fold_generator_losses, avg_disc_loss_cv, avg_gen_loss_cv)

        # After CV, load the generator from the last fold as the 'final' generator
        try:
            last_fold_generator_path = os.path.join(self.config['output_directory'], f'generator_fold_{self.config['n_splits']}.pth')
            if os.path.exists(last_fold_generator_path):
                 print(f"Loading final generator from {last_fold_generator_path}")
                 self.final_generator = Generator(
                    self.config['vocab_size'], self.config['d_model'], self.config['nhead'],
                    self.config['num_layers'], self.config['dim_feedforward'], self.config['dropout']
                ).to(self.device)
                 self.final_generator.load_state_dict(torch.load(last_fold_generator_path, map_location=self.device))
                 self.final_generator.eval()
            else:
                print(f"Could not find generator for the last fold at {last_fold_generator_path}")
                self.final_generator = None
        except Exception as e:
            print(f"Error loading final generator: {e}")
            self.final_generator = None


    def save_model(self, model, file_name):
        """Saves a PyTorch model's state dictionary."""
        save_path = os.path.join(self.config['output_directory'], file_name)
        try:
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}") # Keep print for verification
        except Exception as e:
            print(f"Error saving model to {save_path}: {e}")


    def save_results(self, fold_discriminator_losses, fold_generator_losses, avg_disc_loss_cv, avg_gen_loss_cv):
        """Saves the cross-validation results to a JSON file."""
        results = {
            'fold_discriminator_losses': fold_discriminator_losses,
            'fold_generator_losses': fold_generator_losses,
            'average_discriminator_loss_cv': avg_disc_loss_cv,
            'average_generator_loss_cv': avg_gen_loss_cv,
            'config': self.config
        }
        results_path = os.path.join(self.config['output_directory'], 'cross_validation_results.json')
        try:
            with open(results_path, 'w') as f:
                json.dump(results, f, indent=4)
            print(f"Saved cross-validation results to {results_path}")
        except Exception as e:
            print(f"Error saving cross-validation results to {results_path}: {e}")


    def visualize_grid(self, grid_sequence, ax, title):
        """Visualizes a single flattened grid sequence."""
        sep_token = 11
        pad_token = 0
        start_token = 10

        # Find the separation token to split the input and output grids
        # Remove start and padding tokens before looking for separator
        sequence_no_start_pad = grid_sequence[(grid_sequence != start_token) & (grid_sequence != pad_token)]

        sep_token_indices = (sequence_no_start_pad == sep_token).nonzero(as_tuple=True)[0]

        if sep_token_indices.numel() > 0:
            sep_index_in_no_start_pad = sep_token_indices[0].item()
            # The part after the separator is the generated output attempt
            generated_output_flat = sequence_no_start_pad[sep_index_in_no_start_pad + 1:]

            # Remove padding tokens from the generated output
            generated_output_flat = generated_output_flat[generated_output_flat != pad_token]

            # Try to reshape the output into a square grid for visualization
            size = int(np.sqrt(len(generated_output_flat)))
            if size * size == len(generated_output_flat):
                grid = generated_output_flat.view(size, size).cpu().numpy()
                ax.imshow(grid, cmap='tab10', vmin=0, vmax=9)
                ax.set_title(title)
                ax.set_xticks(np.arange(size))
                ax.set_yticks(np.arange(size))
                ax.set_xticklabels([])
                ax.set_yticklabels([])
                ax.grid(which='both', color='gray', linestyle='-', linewidth=0.5)
            else:
                ax.set_title(f"{title}\n(Non-square grid or reshape error)")
                ax.axis('off')
        else:
            ax.set_title(f"{title}\n(No separator token found)")
            ax.axis('off')


    def generate_and_visualize_samples(self, num_samples=16):
        """
        Loads the final trained generator, generates new samples,
        and visualizes them.
        """
        print("\n--- Generating and Visualizing Samples ---")
        if self.final_generator is None:
            print("Generator not trained or loaded. Cannot generate samples.")
            return

        self.final_generator.eval()
        device = next(self.final_generator.parameters()).device # Get the device from the model
        with torch.no_grad():
            # Generate a batch of random integer indices for the generator's input
            # You could also use actual input grids from the test set here for a more meaningful generation
            fake_input_indices = torch.randint(
                0, self.config['vocab_size'],
                (num_samples, self.config['max_seq_len']),
                dtype=torch.long, device=device
            )
            fake_sequences_logits = self.final_generator(fake_input_indices)

            # Apply Gumbel-Softmax and argmax to get discrete tokens
            # Use a low temperature for sampling closer to argmax
            generated_sequences = torch.argmax(F.gumbel_softmax(fake_sequences_logits, tau=0.1, hard=True, dim=-1), dim=-1)


        # Visualize the generated sequences
        fig, axes = plt.subplots(int(np.sqrt(num_samples)), int(np.sqrt(num_samples)), figsize=(12, 12))
        axes = axes.flatten()
        for i, ax in enumerate(axes):
            self.visualize_grid(generated_sequences[i], ax, f'Generated Grid {i+1}')

        plt.tight_layout()
        plt.show()


# --- 7. REINFORCEMENT LEARNING AGENT ---
class RLAgent(nn.Module):
    def __init__(self, generator, config):
        super().__init__()
        self.generator = generator # Use the trained GAN generator as the policy network
        self.config = config
        self.device = next(generator.parameters()).device # Use the same device as the generator
        self.optimizer = optim.AdamW(self.generator.parameters(), lr=self.config['rl_learning_rate'])
        self.epsilon = self.config['epsilon_start']
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=0) # Ignore padding token in loss

    def forward(self, state):
        # The state is the input sequence for the generator
        return self.generator(state) # Returns logits

    def select_action(self, state):
        # Select action using epsilon-greedy policy
        if random.random() < self.epsilon:
            # Explore: Sample a random action for the entire sequence
            action = torch.randint(0, self.config['vocab_size'], (state.size(0), self.config['max_seq_len']), dtype=torch.long, device=self.device)
        else:
            # Exploit: Sample from the generator's output distribution
            with torch.no_grad():
                logits = self.forward(state)
                # Use Gumbel-Softmax for sampling
                action_one_hot = F.gumbel_softmax(logits, tau=0.1, hard=True, dim=-1)
                action = torch.argmax(action_one_hot, dim=-1)

        return action

    def update_policy(self, states, actions, rewards):
        # Train the generator (policy network) using policy gradient (simplified)
        self.optimizer.zero_grad()
        logits = self.forward(states)

        # Calculate loss - treating this as a sequence generation problem
        # We want the generator to output sequences that lead to high rewards.
        # A simple approach is to use REINFORCE: scale the log probability of the action by the reward.
        # Note: This is a very basic RL setup and might need significant refinement for complex ARC tasks.

        # Flatten sequences and rewards for loss calculation
        logits_flat = logits.view(-1, self.config['vocab_size'])
        actions_flat = actions.view(-1)

        # Calculate log probabilities of the taken actions
        log_probs = F.log_softmax(logits_flat, dim=-1)
        # Gather log probabilities for the taken actions
        taken_action_log_probs = log_probs.gather(1, actions_flat.unsqueeze(1)).squeeze(1)

        # Apply rewards as weights to the log probabilities
        # Need to map rewards back to the sequence elements.
        # Assuming the reward applies to the entire generated sequence:
        rewards_expanded = rewards.unsqueeze(1).expand_as(actions).flatten()

        # Policy gradient loss (negative sign because we want to maximize reward)
        # Multiply by rewards (baseline subtraction could improve stability)
        # Ignore loss where padding token was generated
        mask = (actions_flat != 0) # Mask out padding tokens
        policy_loss = -torch.masked_select(taken_action_log_probs * rewards_expanded, mask).mean()


        policy_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.generator.parameters(), self.config['gradient_clipping_value'])
        self.optimizer.step()

        return policy_loss.item()

    def decay_epsilon(self):
        self.epsilon = max(self.config['epsilon_end'], self.epsilon * self.config['epsilon_decay'])


# --- 8. REINFORCEMENT LEARNING ENVIRONMENT AND REWARD ---
# Define your ARC environment and reward function here.
# This is a crucial part and highly depends on how you represent the ARC tasks
# and how you want to evaluate the generated output.

# Placeholder Environment (needs to be implemented based on ARC task structure)
class ARCEnvironment:
    def __init__(self, test_tasks):
        """
        Args:
            test_tasks (list): A list of test task dictionaries, each containing 'input' and 'output' examples.
        """
        self.test_tasks = test_tasks
        self.current_task_index = 0
        self.current_example_index = 0
        self.current_input_grid = None
        self.current_output_grid = None
        self.start_token = 10
        self.sep_token = 11
        self.pad_token = 0
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


    def reset(self):
        """Resets the environment to a new task/example."""
        if self.current_task_index >= len(self.test_tasks):
            print("All test tasks processed for this environment.")
            return None # Indicate no more tasks

        task = self.test_tasks[self.current_task_index]
        # For RL training, we need input and target output. Use 'train' examples.
        if 'train' in task and task['train']:
             # Use the first training example for the current episode
             train_example = task['train'][0]
             if 'input' in train_example and train_example['input'] and 'output' in train_example and train_example['output']:
                 self.current_input_grid = np.array(train_example['input'])
                 self.current_output_grid = np.array(train_example['output'])

                 # Create the initial state sequence: [start_token, input_flat, sep_token, padding...]
                 # The agent needs to generate the content *after* the sep_token.
                 input_flat = self.current_input_grid.flatten()
                 initial_sequence = np.concatenate([
                     [self.start_token],
                     input_flat,
                     [self.sep_token]
                 ])

                 # Pad to max sequence length
                 padded_initial_sequence = np.pad(initial_sequence, (0, CONFIG['max_seq_len'] - len(initial_sequence)), 'constant', constant_values=self.pad_token)
                 state = torch.tensor(padded_initial_sequence, dtype=torch.long).unsqueeze(0) # Add batch dimension
                 return state.to(self.device) # Return state on device

             else:
                 print(f"Skipping task {self.current_task_index} due to missing input/output in training example.")
                 self.current_task_index += 1
                 return self.reset() # Try the next task
        else:
            print(f"Skipping task {self.current_task_index} due to missing train examples.")
            self.current_task_index += 1
            return self.reset() # Try the next task


    def step(self, action_sequence):
        """
        Takes an action (generated output sequence) and returns the reward and done status.
        Args:
            action_sequence (torch.Tensor): The sequence generated by the agent (batch_size, max_seq_len).
                                             Assumes batch size is 1 for simplicity in this placeholder.
        Returns:
            reward (float): The reward for the action.
            done (bool): True if the episode is finished.
        """
        if action_sequence.size(0) != 1:
             print("Warning: ARCEnvironment step expects batch size 1.")

        generated_sequence = action_sequence.squeeze(0) # Remove batch dimension

        # The agent generates the full sequence, including start/sep/padding.
        # The 'action' conceptually is the part after the separator.
        # To calculate reward, we need to compare the generated output part
        # with the target output grid from the environment's current task.

        reward = 0.0
        done = True # In this simple setup, one action (generating the sequence) ends the episode

        # Extract the generated output grid from the sequence generated by the agent
        # Find the separator token within the *generated* sequence
        sep_token_indices = (generated_sequence == self.sep_token).nonzero(as_tuple=True)[0]

        if sep_token_indices.numel() > 0:
            sep_index = sep_token_indices[0].item()
            # The part after the separator is the generated output attempt
            generated_output_flat = generated_sequence[sep_index + 1:]

            # Remove padding tokens from the generated output
            generated_output_flat = generated_output_flat[generated_output_flat != self.pad_token]

            # --- Reward Calculation ---
            # Compare the generated output_flat with the actual target_output_grid (from train examples)
            if self.current_output_grid is not None:
                 target_output_flat = self.current_output_grid.flatten()

                 min_len = min(len(generated_output_flat), len(target_output_flat))
                 # Compare elements up to the minimum length
                 # Ensure tensors are on the same device
                 # Need to move target_output_flat to the same device as generated_output_flat
                 target_output_tensor = torch.tensor(target_output_flat, dtype=torch.long).to(generated_output_flat.device)
                 matching_elements = torch.sum(generated_output_flat[:min_len] == target_output_tensor[:min_len]).item()

                 # Simple percentage match reward
                 if len(target_output_flat) > 0:
                      reward = matching_elements / len(target_output_flat)
                      # Add a bonus for getting the exact grid shape correct (if reshapeable)
                      target_shape = self.current_output_grid.shape
                      generated_output_len = len(generated_output_flat)
                      if generated_output_len == target_shape[0] * target_shape[1]:
                           # Try reshaping and comparing full grids
                           try:
                                generated_output_grid = generated_output_flat.view(target_shape).cpu().numpy()
                                if np.array_equal(generated_output_grid, self.current_output_grid):
                                     reward = 2.0 # High reward for perfect grid match
                                else:
                                     # Partial match reward if shape is correct but content isn't perfect
                                     reward += 0.5 * (matching_elements / len(target_output_flat)) # Add partial reward for content match
                           except:
                                # Reshape failed, just use flat match reward
                                reward = matching_elements / len(target_output_flat)
                      else:
                          # Use flat match reward if shape is incorrect
                          reward = matching_elements / len(target_output_flat)

                 else: # Target output is empty
                     if len(generated_output_flat) == 0:
                         reward = 1.0 # Perfect match if both are empty
                     else:
                         reward = 0.0 # Generated output when target is empty


        else:
            # Penalty if no separator token is generated
            reward = -1.0 # Example penalty for invalid sequence structure


        # Move to the next task for the next episode
        self.current_task_index += 1

        return reward, done

# --- 9. RL TRAINING LOGIC ---
def train_rl_agent(agent, env, config):
    """Trains the RL agent."""
    print("\n--- Starting RL Agent Training ---")
    os.makedirs(config['rl_output_directory'], exist_ok=True)
    print(f"RL models and results will be saved to: {config['rl_output_directory']}")


    episode_rewards = []
    avg_rewards = []
    policy_losses = []

    start_episode = load_rl_checkpoint(agent, config)

    for episode in range(start_episode, config['rl_num_episodes']):
        state = env.reset() # Get initial state (input grid sequence)

        if state is None: # All tasks processed
            print("All tasks processed for RL training environment.")
            break

        done = False
        total_reward = 0
        episode_policy_losses = []

        # The RL episode consists of the agent generating the output sequence
        # In this simple setup, the agent generates the full sequence in one step
        action = agent.select_action(state) # Generate the full output sequence

        # Step the environment with the generated action
        reward, done = env.step(action) # Get reward based on the generated sequence


        # Train the agent
        # For simplicity, training happens after each episode (generating one sequence)
        if episode >= config['rl_warmup_episodes']: # Start training after some warmup episodes
            policy_loss = agent.update_policy(state, action, torch.tensor([reward], dtype=torch.float, device=agent.device))
            episode_policy_losses.append(policy_loss)


        total_reward += reward

        episode_rewards.append(total_reward)
        agent.decay_epsilon() # Decay epsilon after each episode

        if episode_policy_losses:
            policy_losses.append(sum(episode_policy_losses) / len(episode_policy_losses))
        else:
            policy_losses.append(0) # No training during warmup

        # Print progress
        if (episode + 1) % 10 == 0 or episode == start_episode: # Print more frequently initially
            avg_reward_last_10 = sum(episode_rewards[-10:]) / 10 if len(episode_rewards) >= 10 else sum(episode_rewards) / len(episode_rewards) if episode_rewards else 0
            print(f"Episode {episode+1}/{config['rl_num_episodes']}, Avg Reward (last 10): {avg_reward_last_10:.4f}, Epsilon: {agent.epsilon:.4f}, Loss: {policy_losses[-1]:.4f}")

        if (episode + 1) % 100 == 0: # Print avg reward over 100 episodes
            avg_reward_last_100 = sum(episode_rewards[-100:]) / 100 if len(episode_rewards) >= 100 else sum(episode_rewards) / len(episode_rewards) if episode_rewards else 0
            avg_rewards.append(avg_reward_last_100)
            print(f"--- Episode {episode+1}/{config['rl_num_episodes']}, Avg Reward (last 100): {avg_reward_last_100:.4f} ---")


        # Save RL checkpoint
        if (episode + 1) % config['rl_checkpoint_interval'] == 0:
            save_rl_checkpoint(agent, episode + 1, config)


    print("\n--- RL Agent Training Finished ---")

    # Save final RL agent state (the updated generator)
    save_rl_model(agent.generator, 'final_rl_generator.pth', config)

    # Save RL training results
    rl_results = {
        'episode_rewards': episode_rewards,
        'avg_rewards_last_100': avg_rewards,
        'policy_losses': policy_losses,
        'config': config
    }
    rl_results_path = os.path.join(config['rl_output_directory'], 'rl_training_results.json')
    with open(rl_results_path, 'w') as f:
        json.dump(rl_results, f, indent=4)
    print(f"RL training results saved to {rl_results_path}")

    # Plot RL training results
    plt.figure(figsize=(12, 6))
    plt.plot(episode_rewards)
    plt.title('RL Episode Rewards')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.grid(True)
    plt.savefig(os.path.join(config['rl_output_directory'], 'rl_episode_rewards.png'))
    # plt.show() # Don't show plots in writefile
    plt.close()

    if avg_rewards:
        plt.figure(figsize=(12, 6))
        # Adjust x-axis for avg_rewards to correspond to the correct episodes
        avg_reward_episodes = list(range(100, start_episode + len(avg_rewards)*100 + 1, 100))
        plt.plot(avg_reward_episodes, avg_rewards)
        plt.title('RL Average Reward (Last 100 Episodes)')
        plt.xlabel('Episode')
        plt.ylabel('Average Reward')
        plt.grid(True)
        plt.savefig(os.path.join(config['rl_output_directory'], 'rl_avg_rewards.png'))
        plt.close()

    if policy_losses:
        plt.figure(figsize=(12, 6))
        # Policy loss corresponds to episodes after warmup
        policy_loss_episodes = list(range(config['rl_warmup_episodes'], start_episode + len(policy_losses) + 1))
        plt.plot(policy_loss_episodes, policy_losses)
        plt.title('RL Policy Loss')
        plt.xlabel('Episode')
        plt.ylabel('Loss')
        plt.grid(True)
        plt.savefig(os.path.join(config['rl_output_directory'], 'rl_policy_loss.png'))
        plt.close()


def save_rl_model(model, file_name, config):
    """Saves the RL model's state dictionary."""
    save_path = os.path.join(config['rl_output_directory'], file_name)
    try:
        torch.save(model.state_dict(), save_path)
        print(f"RL model saved to {save_path}")
    except Exception as e:
        print(f"Error saving RL model to {save_path}: {e}")


def save_rl_checkpoint(agent, episode, config):
    """Saves an RL training checkpoint."""
    checkpoint_path = os.path.join(config['rl_output_directory'], f'rl_checkpoint_episode_{episode}.pth')
    try:
        torch.save({
            'episode': episode,
            'generator_state_dict': agent.generator.state_dict(),
            'optimizer_state_dict': agent.optimizer.state_dict(),
            'epsilon': agent.epsilon,
        }, checkpoint_path)
        print(f"RL checkpoint saved for Episode {episode} to {checkpoint_path}") # Keep print for verification
    except Exception as e:
        print(f"Error saving RL checkpoint to {checkpoint_path}: {e}")


def load_rl_checkpoint(agent, config):
    """Loads the latest RL checkpoint."""
    checkpoint_files = glob.glob(os.path.join(config['rl_output_directory'], 'rl_checkpoint_episode_*.pth'))
    if not checkpoint_files:
        print("No RL checkpoint found. Starting RL training from scratch.")
        return 0

    latest_checkpoint = max(checkpoint_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
    print(f"Loading RL checkpoint: {latest_checkpoint}")

    try:
        checkpoint = torch.load(latest_checkpoint, map_location=agent.device)
        agent.generator.load_state_dict(checkpoint['generator_state_dict'])
        agent.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        agent.epsilon = checkpoint['epsilon']

        # Remove older checkpoints
        for old_checkpoint in checkpoint_files:
            if old_checkpoint != latest_checkpoint:
                 try:
                     os.remove(old_checkpoint)
                     # print(f"Removed old RL checkpoint: {old_checkpoint}") # Commented out
                 except OSError as e:
                    print(f"Error removing old RL checkpoint {old_checkpoint}: {e}")


        return checkpoint['episode']
    except Exception as e:
        print(f"Error loading RL checkpoint {latest_checkpoint}: {e}")
        print("Starting RL training from scratch.")
        return 0


# --- 10. KAGGLE SUBMISSION GENERATION ---
def generate_submission_file(generator, config):
    """Generates the Kaggle submission file."""
    print("\n--- Generating Submission File ---")
    generator.eval()
    device = next(generator.parameters()).device

    test_data_path = Path(config['base_data_path']) / 'GridTransitionDataset/test' # Assuming test data is in a 'test' subdir
    test_task_files = glob.glob(os.path.join(test_data_path, '*.json'))

    submission = []

    if not test_task_files:
        print(f"No test files found in {test_data_path}. Cannot generate submission.")
        return

    start_token = 10
    sep_token = 11
    pad_token = 0

    for task_file in tqdm(test_task_files, desc="Generating Submissions"):
        try:
            with open(task_file, 'r') as f:
                task = json.load(f)
                task_id = Path(task_file).stem

                if 'test' in task and task['test']:
                    for i, test_example in enumerate(task['test']):
                        if 'input' in test_example and test_example['input']:
                            input_grid = np.array(test_example['input'])
                            input_flat = input_grid.flatten()

                            # Create the input sequence for the generator: [start_token, input_flat, sep_token, padding...]
                            input_sequence = np.concatenate([
                                [start_token],
                                input_flat,
                                [sep_token]
                            ])

                            # Pad to max sequence length
                            padded_input_sequence = np.pad(input_sequence, (0, config['max_seq_len'] - len(input_sequence)), 'constant', constant_values=pad_token)
                            input_tensor = torch.tensor(padded_input_sequence, dtype=torch.long).unsqueeze(0).to(device) # Add batch dimension and move to device

                            # Generate the output sequence using the trained generator
                            with torch.no_grad():
                                logits = generator(input_tensor)
                                # Use argmax for the final output sequence in submission
                                generated_sequence = torch.argmax(logits, dim=-1).squeeze(0) # Remove batch dimension

                            # Extract the generated output grid
                            sep_token_indices = (generated_sequence == sep_token).nonzero(as_tuple=True)[0]

                            predicted_output = []
                            if sep_token_indices.numel() > 0:
                                sep_index = sep_token_indices[0].item()
                                # The part after the separator is the generated output attempt
                                generated_output_flat = generated_sequence[sep_index + 1:]

                                # Remove padding tokens
                                generated_output_flat = generated_output_flat[generated_output_flat != pad_token]

                                # Try to reshape the flat output into a grid
                                # This is tricky without knowing the target dimensions.
                                # A simple approach is to assume square or infer from the input size if possible.
                                # For a valid submission, we need to output a list of lists representing the 2D grid.
                                # A very basic attempt: if the flattened output length suggests a square, reshape it.
                                output_len = len(generated_output_flat)
                                size = int(np.sqrt(output_len))
                                if size * size == output_len:
                                    predicted_output_grid = generated_output_flat.view(size, size).cpu().tolist()
                                    predicted_output = predicted_output_grid
                                else:
                                     # If reshaping fails, return an empty list or a default structure
                                     # For a real solution, you'd need a more robust way to determine output shape.
                                     print(f"Warning: Could not reshape generated output for task {task_id}, example {i}. Output length: {output_len}")
                                     predicted_output = [] # Or some other default

                            else:
                                # No separator token, likely an invalid generation
                                predicted_output = [] # Return empty list

                            submission.append({
                                'task_id': task_id,
                                'output_id': i,
                                'output': predicted_output
                            })
                        else:
                             print(f"Skipping test example {i} in task {task_id} due to missing input.")

                else:
                    print(f"Skipping task {task_id} due to missing test examples.")

        except json.JSONDecodeError:
            print(f"Error decoding JSON from test file: {task_file}")
            continue
        except Exception as e:
            print(f"Error processing test file {task_file}: {e}")
            continue

    # Save the submission file
    submission_path = config['submission_file']
    try:
        with open(submission_path, 'w') as f:
            json.dump(submission, f, indent=4)
        print(f"Submission file generated at {submission_path}")
    except Exception as e:
        print(f"Error saving submission file to {submission_path}: {e}")


# --- 11. MAIN EXECUTION ---
if __name__ == '__main__':
    # Mount Google Drive (if running in Colab)
    # from google.colab import drive
    # drive.mount('/content/drive')

    # Preprocess and cache data
    all_sequences = preprocess_and_cache_data(CONFIG)

    if not all_sequences:
        print("No processed data available. Exiting.")
    else:
        # --- GAN Training ---
        print("\n--- Initializing GAN Trainer ---")
        gan_trainer = GANTrainer(CONFIG, all_sequences)
        gan_trainer.run_cross_validation()

        # After GAN training, the final_generator in gan_trainer should hold the last trained model
        final_gan_generator = gan_trainer.final_generator

        if final_gan_generator is not None:
            # Visualize some samples from the GAN generator
            gan_trainer.generate_and_visualize_samples()

            # --- Reinforcement Learning Training ---
            print("\n--- Initializing RL Training ---")
            # Load original tasks for the RL environment
            print("Loading original task data for RL environment...")
            # Need to load tasks from the *training* set for RL training environment
            rl_data_path = os.path.join(CONFIG['base_data_path'], CONFIG['input_directory'])
            all_task_files = glob.glob(
                os.path.join(rl_data_path, '**', '*.json'),
                recursive=True
            )
            all_tasks = []
            for task_file in tqdm(all_task_files, desc="Loading Tasks for RL Env"):
                 try:
                     with open(task_file, 'r') as f:
                         all_tasks.append(json.load(f))
                 except json.JSONDecodeError:
                     print(f"Error decoding JSON from file: {task_file}")
                     continue


            if not all_tasks:
                 print("No original task data loaded for RL environment. Skipping RL training.")
            else:
                # Use loaded tasks for the RL training environment
                # Filter tasks to ensure they have training examples with input/output for the environment
                filtered_tasks = [
                    task for task in all_tasks
                    if 'train' in task and task['train'] and
                       'input' in task['train'][0] and task['train'][0]['input'] and
                       'output' in task['train'][0] and task['train'][0]['output']
                ]
                if not filtered_tasks:
                    print("No suitable training tasks found for RL environment. Skipping RL training.")
                else:
                    print(f"Using {len(filtered_tasks)} tasks for RL training environment.")
                    rl_env = ARCEnvironment(filtered_tasks)

                    # Initialize the RL agent with the trained GAN generator
                    rl_agent = RLAgent(final_gan_generator, CONFIG)

                    # Train the RL agent
                    train_rl_agent(rl_agent, rl_env, CONFIG)

                    # --- Kaggle Submission ---
                    # The train_rl_agent function saves the final_rl_generator.
                    # Load this generator to make predictions on the *actual* test set.
                    final_rl_generator_path = os.path.join(CONFIG['rl_output_directory'], 'final_rl_generator.pth')
                    if os.path.exists(final_rl_generator_path):
                         print("\n--- Loading Final RL Generator for Submission ---")
                         submission_generator = Generator(
                            CONFIG['vocab_size'], CONFIG['d_model'], CONFIG['nhead'],
                            CONFIG['num_layers'], CONFIG['dim_feedforward'], CONFIG['dropout']
                        ).to(gan_trainer.device) # Use the same device as the trainer
                         submission_generator.load_state_dict(torch.load(final_rl_generator_path, map_location=gan_trainer.device))

                         # Generate the submission file
                         generate_submission_file(submission_generator, CONFIG)
                    else:
                        print(f"Could not find final RL generator model at {final_rl_generator_path}. Cannot generate submission.")

        else:
             print("GAN Generator was not successfully trained or loaded. Skipping RL training and submission generation.")


    print("\n--- Overall process finished. ---")

**Reasoning**:
The script `arc_gan_rl_solution.py` has been updated with the corrected data path and additional debugging prints. The next step is to execute this script to see if the data loading is successful and if the GAN training and RL implementation can proceed.



In [ ]:
!python arc_gan_rl_solution.py

## Summary:

### Data Analysis Key Findings

*   The Python script `arc_gan_rl_solution.py` was created to handle the entire workflow, including data loading, preprocessing, GAN training, RL training, and submission generation.
*   The script consistently failed during the data loading phase across multiple attempts due to a `FileNotFoundError`.
*   The error message indicated that the specified data directory path (`/content/drive/MyDrive/Google_AI_Studio/ARCAGI2025/data/GridTransitionDataset/training_transformed_unique_ids` or `/content/drive/MyDrive/ARCAGI2025/data/GridTransitionDataset/training_transformed_unique_ids` in a subsequent attempt) could not be found.
*   Debugging print statements confirmed the data path being used by the script, which did not resolve the file not found issue.
*   Attempts to delete a potentially empty processed data cache file were made but did not resolve the fundamental data access problem.
*   The GAN training, RL implementation, RL training, and submission generation steps were never reached because the initial data loading failed.

### Insights or Next Steps

*   The user needs to verify the exact path where the `GridTransitionDataset/training_transformed_unique_ids` directory is located on their Google Drive and update the `base_data_path` and `input_directory` in the `CONFIG` dictionary accordingly.
*   Ensure Google Drive is correctly mounted and accessible in the execution environment.
